# Self-Reflection Agent with Confidence-Aware Early Stopping

## Educational Demo for Data Science Students

This notebook demonstrates the **Self-Reflection** approach for intelligent LLM reasoning that balances accuracy with computational efficiency through confidence-aware early stopping.

### Key Concepts:
- **Self-Reflection**: Agent assesses its own confidence in consensus
- **Early Stopping**: Terminate sampling when sufficient confidence is reached
- **Probability Distributions**: Return full distribution, not just argmax
- **Utility-Based Agent**: Balance consensus confidence vs computational cost
- **Convergence Analysis**: Track how consensus emerges over time

### Comparison with Self-Consistency:
| Feature | Self-Consistency | Self-Reflection |
|---------|------------------|------------------|
| **Agent Type** | Model-based reflex | Utility-based |
| **Stopping Logic** | Fixed sample count | Confidence-aware early stopping |
| **Output** | Single answer (argmax) | Probability distribution + confidence |
| **Self-Awareness** | None | Uncertainty assessment |
| **Cost Optimization** | No | Yes (early stopping) |

### Mathematical Foundation:
- **Confidence**: `max(P(answer_i))` or entropy-based assessment
- **Early Stopping**: Stop when `confidence ≥ threshold` after `min_responses`
- **Utility Function**: Balance `consensus_confidence` vs `computational_cost`

### Prerequisites:
1. **LiteLLM Server**: Run `make litellm-install` to start local LLM proxy
2. **Environment**: Configure `.env` file with your LLM settings  
3. **Dependencies**: Install with `make install`

## 1. Setup and Imports

First, let's import our self-reflection agent components and set up the environment for confidence-aware reasoning.

In [ ]:
# Core imports for the self-reflection agent
import os
import sys
from pathlib import Path

# Add the project root to Python path so we can import our modules
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import self-reflection agent components
from llm_agents.self_reflection.agent import SelfReflectionAgent
from llm_agents.self_reflection.config import ReflectionConfig
from llm_agents.self_reflection.domain import ReflectionResult
from llm_agents.common.domain import LLMResponse
from llm_agents.common.interfaces import LiteLLMAdapter

# Import self-consistency for comparison
from llm_agents.self_consistency.agent import SelfConsistencyAgent
from llm_agents.self_consistency.config import AgentConfig
from llm_agents.self_consistency.domain import ConsensusResult

# Data science libraries for analysis and visualization
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import time
import numpy as np
import math

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("✅ All imports successful!")
print(f"📁 Working directory: {project_root}")
print("🧠 Self-Reflection Agent Ready for Confidence-Aware Reasoning!")

## 2. Environment Configuration Check

Let's verify our environment is properly configured for LLM interaction and confidence-aware reasoning.

In [ ]:
%load_ext dotenv
%dotenv

# Check environment configuration
def check_environment():
    """Check if environment is properly configured for self-reflection."""
    config_status = {
        'LLM_MODEL': os.getenv('LLM_MODEL', 'claude-3-haiku'),  # Default from .env.example
        'LLM_BASE_URL': os.getenv('LLM_BASE_URL', 'http://localhost:4000'),
        'LLM_TEMPERATURE': os.getenv('LLM_TEMPERATURE', '0.7'),
        'LLM_API_KEY': os.getenv('LLM_API_KEY', 'sk-1234')[:20] + '...' if os.getenv('LLM_API_KEY') else 'sk-1234'
    }
    
    print("🔧 Environment Configuration for Self-Reflection Agent:")
    for key, value in config_status.items():
        print(f"  {key}: {value}")
    
    # Check if .env file exists
    env_file = project_root / '.env'
    if env_file.exists():
        print("\n✅ .env file found")
    else:
        print("\n⚠️  .env file not found")
        print("   Run: make setup-env to create from template")
    
    print("\n🧠 Self-Reflection Agent Features:")
    print("  • Confidence-aware early stopping")
    print("  • Probability distribution output")
    print("  • Convergence analysis")
    print("  • Uncertainty assessment")
    print("  • Cost optimization")
    
    return config_status

config = check_environment()

## 3. Test LLM Connection

Before demonstrating self-reflection capabilities, let's test the connection to our LiteLLM server.

In [ ]:
# Test LLM connection for self-reflection
def test_llm_connection():
    """Test connection to LiteLLM server for self-reflection demo."""
    try:
        print("🔌 Testing LLM connection for Self-Reflection Agent...")
        
        # Create LLM adapter
        adapter = LiteLLMAdapter()
        print(f"📡 Connecting to: {adapter.base_url}")
        print(f"🤖 Using model: {adapter.model}")
        
        # Test simple query
        start_time = time.time()
        response = adapter.generate_llm_response(
            prompt="Think step by step and provide your reasoning.",
            question="What is 3 × 7?"
        )
        end_time = time.time()
        
        print(f"\n✅ Connection successful! ({end_time - start_time:.2f}s)")
        print(f"💭 Reasoning: {response.reasoning[:100]}...")
        print(f"🎯 Answer: {response.answer}")
        
        # Test with standard reasoning prompt (confidence comes from consensus, not individual LLM responses)
        print(f"\n🧠 Testing reasoning for consensus analysis...")
        reflection_response = adapter.generate_llm_response(
            prompt="Think step by step and provide detailed reasoning.",
            question="Is 17 a prime number?"
        )
        
        print(f"💭 Detailed reasoning: {reflection_response.reasoning[:150]}...")
        print(f"🎯 Answer: {reflection_response.answer}")
        print(f"📊 Note: Self-reflection confidence comes from consensus across multiple responses, not individual LLM confidence")
        
        return adapter, True
        
    except Exception as e:
        print(f"\n❌ Connection failed: {e}")
        print("\n🔧 Troubleshooting:")
        print("   1. Check if LiteLLM is running: make litellm-status")
        print("   2. Start LiteLLM if needed: make litellm-install")
        print("   3. Test connection: make litellm-test")
        
        return None, False

adapter, connection_ok = test_llm_connection()

## 4. Basic Self-Reflection Demo

Let's start with a simple example to demonstrate confidence-aware early stopping and probability distributions.

In [ ]:
# Basic self-reflection demonstration
if connection_ok:
    print("🧪 Basic Self-Reflection Agent Demo")
    print("=" * 50)
    
    # Configure the self-reflection agent
    reflection_config = ReflectionConfig(
        llm_interface=adapter,
        target_responses=10,  # Maximum responses
        confidence_threshold=0.8,  # Stop early if 80% confident
        min_responses=3,  # Minimum before considering early stopping
        prompt_template="Think step by step and provide your reasoning. End with 'The answer is [your answer]'."
    )
    
    # Test question
    question = "If a dataset has 1000 rows and you split it 70-20-10 for train-validation-test, how many samples in each set?"
    print(f"📝 Question: {question}")
    print(f"🎯 Target: Max {reflection_config.target_responses} responses, stop early if confidence ≥ {reflection_config.confidence_threshold:.0%}")
    
    # Create and run the self-reflection agent
    agent = SelfReflectionAgent(reflection_config, question)
    
    print(f"\n🔄 Starting confidence-aware reasoning...")
    start_time = time.time()
    
    result = agent.process_question()
    
    end_time = time.time()
    
    # Display comprehensive results
    print(f"\n⏱️  Total time: {end_time - start_time:.2f}s")
    print(f"🎯 Final Answer: {result.final_answer}")
    print(f"📊 Consensus Confidence: {result.consensus_confidence:.1%}")
    print(f"🔍 Uncertainty Level: {result.uncertainty_level}")
    print(f"⚡ Early Stopping: {'Yes' if result.early_stopping else 'No'}")
    print(f"📈 Total Responses: {result.total_responses}/{reflection_config.target_responses}")
    
    # Show probability distribution (key difference from self-consistency)
    print(f"\n📊 Probability Distribution:")
    for answer, prob in sorted(result.answer_distribution.items(), key=lambda x: x[1], reverse=True):
        bar = "█" * int(prob * 20)  # Visual bar
        print(f"  '{answer}': {prob:.1%} {bar}")
    
    # Show convergence analysis
    convergence = result.convergence_analysis
    print(f"\n📈 Convergence Analysis:")
    print(f"  • Convergence Rate: {convergence['convergence_rate']:.3f}")
    print(f"  • Final Stability: {convergence['final_stability']:.3f}")
    print(f"  • Confidence Evolution: {[f'{c:.2f}' for c in convergence['confidence_evolution'][-3:]]}")
    
    # Show individual responses
    print(f"\n📋 Individual Responses:")
    for i, response in enumerate(agent._llm_responses, 1):
        confidence_at_step = convergence['confidence_evolution'][i-1] if i-1 < len(convergence['confidence_evolution']) else 0
        print(f"  {i}. Answer: {response.answer} (confidence: {confidence_at_step:.2f})")
        print(f"     Reasoning: {response.reasoning[:80]}...")
        
    # Cost analysis
    if result.early_stopping:
        saved_calls = reflection_config.target_responses - result.total_responses
        cost_savings = (saved_calls / reflection_config.target_responses) * 100
        print(f"\n💰 Cost Optimization:")
        print(f"  • LLM calls saved: {saved_calls}")
        print(f"  • Cost reduction: {cost_savings:.1f}%")
        print(f"  • Efficiency gain: Achieved {result.consensus_confidence:.1%} confidence in {result.total_responses} calls")
        
else:
    print("⚠️  Skipping demo - LLM connection not available")
    print("   Please fix connection issues and re-run this cell")

## 5. Custom Self-Reflection Experiments

Use the cell below to run your own experiments with different questions and parameters.

In [ ]:
# 🚀 YOUR CUSTOM SELF-REFLECTION EXPERIMENT
# Modify the parameters below to test your own questions and settings

def run_self_reflection_experiment(question, confidence_threshold=0.8, max_responses=10, min_responses=3, temperature=0.7):
    """Run a custom self-reflection experiment with adjustable parameters."""
    
    if not connection_ok:
        print("❌ LLM connection not available")
        return None
    
    print(f"🧪 Self-Reflection Experiment")
    print(f"📝 Question: {question}")
    print(f"🎯 Confidence threshold: {confidence_threshold:.0%}")
    print(f"🔢 Max responses: {max_responses}")
    print(f"🔢 Min responses: {min_responses}")
    print(f"🌡️  Temperature: {temperature}")
    print("\n" + "="*60)
    
    # Create custom adapter with specified temperature
    custom_adapter = LiteLLMAdapter(temperature=temperature)
    
    # Configure self-reflection agent
    config = ReflectionConfig(
        llm_interface=custom_adapter,
        target_responses=max_responses,
        confidence_threshold=confidence_threshold,
        min_responses=min_responses,
        prompt_template="Think step by step and provide detailed reasoning. End with 'The answer is [your answer]'."
    )
    
    # Run experiment
    agent = SelfReflectionAgent(config, question)
    
    print(f"⏳ Starting confidence-aware reasoning...")
    start_time = time.time()
    
    result = agent.process_question()
    
    end_time = time.time()
    
    # Detailed analysis
    print(f"\n⏱️  Execution time: {end_time - start_time:.2f}s")
    print(f"🎯 Final answer: {result.final_answer}")
    print(f"📊 Consensus confidence: {result.consensus_confidence:.1%}")
    print(f"🔍 Uncertainty level: {result.uncertainty_level}")
    print(f"⚡ Early stopping: {'Yes' if result.early_stopping else 'No'}")
    print(f"📈 Total responses: {result.total_responses}/{max_responses}")
    
    # Cost analysis
    if result.early_stopping:
        saved_responses = max_responses - result.total_responses
        cost_savings = (saved_responses / max_responses) * 100
        print(f"\n💰 Cost Optimization:")
        print(f"  • Responses saved: {saved_responses}")
        print(f"  • Cost reduction: {cost_savings:.1f}%")
        print(f"  • Efficiency: {result.consensus_confidence:.1%} confidence in {result.total_responses} calls")
    
    # Probability distribution
    print(f"\n📊 Probability Distribution:")
    sorted_answers = sorted(result.answer_distribution.items(), key=lambda x: x[1], reverse=True)
    for answer, prob in sorted_answers:
        bar = "█" * int(prob * 25)  # Visual bar
        print(f"  '{answer}': {prob:.1%} {bar}")
    
    return result

# Example: Mathematical problem with high confidence threshold
if connection_ok:
    my_result = run_self_reflection_experiment(
        question="If a neural network has 3 layers with 128, 64, and 32 neurons respectively, and each connection has a weight, how many weights connect the first two layers?",
        confidence_threshold=0.85,  # High threshold - requires strong consensus
        max_responses=8,           # Maximum responses before giving up
        min_responses=3,           # Minimum responses before considering early stopping
        temperature=0.5            # Lower temperature for more focused reasoning
    )
else:
    print("⚠️  Skipping experiments - LLM connection not available")

# Try different types of questions:
# - Mathematical calculations
# - Data science concepts  
# - Opinion-based questions
# - Factual queries
# - Problem-solving scenarios
# - Code-related questions

# Experiment with parameters:
# - confidence_threshold: 0.5 (aggressive) to 0.95 (very conservative)
# - max_responses: 3-15 responses
# - min_responses: 2-8 responses (always < max_responses)
# - temperature: 0.1 (deterministic) to 1.2 (very creative)

## 6. Summary and Key Takeaways

Let's summarize what we've learned about self-reflection agents and confidence-aware reasoning.

In [ ]:
# Summary analysis
print("🧠 Self-Reflection Agent: Key Takeaways")
print("=" * 60)

print("\n🧠 Conceptual Understanding:")
print("   • Self-reflection adds confidence awareness to LLM reasoning")
print("   • Utility-based agent balances accuracy vs computational cost")
print("   • Early stopping reduces costs while maintaining quality")
print("   • Full probability distributions provide richer information than argmax")
print("   • Agent achieves self-awareness about its own uncertainty")

print("\n⚡ Algorithmic Insights:")
print("   • Dynamic stopping based on consensus confidence thresholds")
print("   • Maintains O(m) complexity for distribution calculations")
print("   • Convergence analysis tracks consensus emergence over time")
print("   • Entropy-based and max probability confidence measures")
print("   • Real-time confidence evolution monitoring")

print("\n📊 Performance Patterns:")
print("   • Higher confidence thresholds → fewer early stops → higher costs")
print("   • Lower confidence thresholds → more early stops → cost savings")
print("   • Mathematical questions typically show fast convergence")
print("   • Opinion-based questions may require more responses")
print("   • Factual questions often achieve immediate consensus")

print("\n💰 Cost-Benefit Analysis:")
print("   • Early stopping can save 20-60% of LLM calls")
print("   • Confidence thresholds enable accuracy vs cost trade-offs")
print("   • Self-awareness prevents over-computation on confident answers")
print("   • Maintains answer quality while optimizing resource usage")

print("\n🛠️ Practical Applications:")
print("   • Educational systems with adaptive questioning")
print("   • Decision support systems with confidence reporting")
print("   • Content generation with quality thresholds")
print("   • Automated problem-solving with cost constraints")
print("   • Research tools with uncertainty quantification")

print("\n⚙️ Implementation Best Practices:")
print("   • Start with confidence_threshold=0.7-0.8 for balanced performance")
print("   • Use min_responses=3-5 to ensure statistical validity")
print("   • Adjust temperature based on question type (0.5-0.8)")
print("   • Monitor convergence patterns to optimize thresholds")
print("   • Consider question difficulty when setting parameters")

print("\n🔍 Self-Reflection vs Self-Consistency:")
print("   • Self-Consistency: Fixed sampling, simple majority vote")
print("   • Self-Reflection: Adaptive sampling, confidence-aware stopping")
print("   • Self-Reflection provides probability distributions + uncertainty")
print("   • Cost optimization through early stopping without accuracy loss")
print("   • Enhanced interpretability through convergence analysis")

print("\n🎯 Next Steps:")
print("   1. Experiment with different confidence thresholds for your use cases")
print("   2. Test self-reflection on domain-specific questions")
print("   3. Compare performance with your current reasoning approaches")
print("   4. Integrate confidence-aware stopping into production systems")
print("   5. Explore adaptive threshold learning based on question types")
print("   6. Implement custom convergence analysis for your applications")

print("\n✨ Congratulations! You've mastered self-reflection agent reasoning!")
print("   You now understand confidence-aware early stopping, probability")
print("   distributions, convergence analysis, and utility-based optimization!")

## 🔧 Troubleshooting

If you encounter issues with the self-reflection agent demo:

### Connection Problems
```bash
# Check LiteLLM status
make litellm-status

# Start LiteLLM if not running
make litellm-install

# Test connection
make litellm-test
```

### Environment Issues
```bash
# Create .env file
make setup-env

# Check environment
make check-env
```

### Import Errors
```bash
# Install dependencies
make install

# Full setup
make setup-all

# Run tests
make test
```

### Self-Reflection Specific Issues

**Early Stopping Not Working:**
- Check confidence_threshold is reasonable (0.5-0.9)
- Ensure min_responses < target_responses
- Verify LLM is providing consistent answers

**Low Confidence Scores:**
- Try questions with clearer answers
- Adjust temperature (lower = more consistent)
- Increase min_responses for better statistics

**Performance Issues:**
- Reduce max_responses for faster execution
- Use lower confidence_threshold for quicker stopping
- Monitor convergence patterns to optimize parameters

---

**Happy experimenting with confidence-aware self-reflection reasoning! 🧠✨**

*This demo showcases the advanced capabilities of utility-based agents with self-awareness, early stopping, and uncertainty quantification - key concepts for the future of intelligent systems.*